<div class="blog-language-switch" role="group" aria-label="文章语言">
<a href="../../Deep-Learning/04-backpropagation-automatic-differentiation.html" lang="en" hreflang="en">English</a>
<span aria-current="page">中文</span>
</div>

[返回深度学习总览](Deep-Learning.html)

## **反向传播与自动微分** {#backpropagation-automatic-differentiation}

只有当最终目标能够沿依赖关系回溯到所有曾影响它的参数时，神经网络才有可能学习。**反向传播（backpropagation）**是在这张依赖图上反向遍历，并高效应用链式法则的过程。**自动微分（automatic differentiation, AD）**则是一种更广义的软件技术：它把程序分解成可微的基本操作，再机械地组合这些操作的导数规则。在常规深度学习训练中，框架使用反向模式自动微分，从标量损失出发完成反向传播。

这些术语彼此相关，但并不完全等价。反向传播描述梯度如何穿过分层模型或图结构模型；反向模式自动微分是一种通用算法模式；PyTorch autograd 则是它的一种具体实现，并明确规定了计算图构建、保存张量、梯度累加和高阶微分等行为。

本章将深入 `.backward()` 之下，回答五个实际问题：

1. 梯度能够说明参数的什么性质，又不能说明什么？
2. 当计算图发生分支并再次汇合时，局部导数规则如何组合？
3. 为什么反向模式不需要构造巨大的 Jacobian 矩阵？
4. 前向传播中必须保留哪些信息？
5. 如何发现错误、消失、爆炸或断开的梯度？

第 05 章将讨论如何用这些梯度进行优化。本章关注的是梯度本身是否正确，以及它在数值计算中的行为。

### **作为信用分配的学习** {#learning-credit-assignment}

设参数为 $\theta \in \mathbb{R}^{P}$ 的模型产生预测 $f_\theta(X)$，并由一个标量损失度量预测与目标之间的差异：

$$
\mathcal{L}(\theta)=\ell(f_\theta(X),Y).
$$

**信用分配问题（credit-assignment problem）**询问：每个参数对当前损失贡献了多少，以及沿哪个局部方向改变该参数会使损失发生怎样的变化。答案由梯度给出：

$$
\nabla_\theta \mathcal{L}
=
\begin{bmatrix}
\partial \mathcal{L}/\partial \theta_1 & \cdots & \partial \mathcal{L}/\partial \theta_P
\end{bmatrix}^{\top}.
$$

对于一个很小的扰动 $\Delta\theta$，一阶 Taylor 展开给出

$$
\mathcal{L}(\theta+\Delta\theta)
\approx
\mathcal{L}(\theta)
+
\nabla_\theta\mathcal{L}^{\top}\Delta\theta.
$$

因此，在 Euclidean 范数下，梯度指向局部上升最快的方向；当梯度非零时，$-\nabla_\theta\mathcal{L}$ 是一个局部下降方向。某个梯度分量为正，并不表示相应参数“有问题”。它只表示：在其他坐标保持不变时，增大这个坐标会使当前损失在局部增大。若重新参数化、改变尺度、换一个批次或更换目标函数，这种解释也会随之改变。

信用分配也不等于因果解释。参数梯度描述的是指定计算过程的局部敏感度。它本身不能证明某个输入特征导致了真实世界中的结果，也不能证明某个参数具有明确语义，更不能保证一次有限幅度的更新一定产生线性近似所预测的效果。

<details>
<summary><strong>PyTorch：用梯度检验局部下降方向</strong></summary>

~~~python
import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(31)

model = nn.Sequential(nn.Linear(3, 5), nn.Tanh(), nn.Linear(5, 1))
features = torch.randn(8, 3)
targets = torch.randn(8, 1)

loss_before = F.mse_loss(model(features), targets)
model.zero_grad()
loss_before.backward()

# Every trainable parameter should receive a finite gradient from this loss.
for name, parameter in model.named_parameters():
    assert parameter.grad is not None, f"missing gradient: {name}"
    assert torch.isfinite(parameter.grad).all(), f"non-finite gradient: {name}"

# Take a deliberately small step along the negative gradient.
step_size = 1e-3
with torch.no_grad():
    for parameter in model.parameters():
        parameter.add_(parameter.grad, alpha=-step_size)

loss_after = F.mse_loss(model(features), targets)
print("loss before:", float(loss_before.detach()))
print("loss after: ", float(loss_after.detach()))
assert loss_after < loss_before
~~~

</details>

这个实验验证的是一个局部命题，而不是一种完整的训练策略。更大的步长可能越过合适区域；在一个 mini-batch 上计算的梯度，也未必降低总体损失。动量、自适应缩放、学习率调度和随机性都会改变“信用”如何转化为参数更新；这些机制属于优化，而不是微分本身。

**应用。** 梯度可用于训练参数、发现与损失断开的模块、进行敏感度分析，并为对抗扰动、影响近似、元学习和可微模拟等方法提供基础组件。

**对比总结。** 损失为完整预测分配一个标量分数；梯度把局部敏感度分配给各个参数；优化器决定如何把这种敏感度转化为有限幅度的更新。三者都不能单独提供因果解释。

### **计算图上的链式法则** {#chain-rule-computation-graphs}

复杂模型的求导可以通过把它分解成简单操作来完成。若

$$
u=g(x), \qquad y=f(u),
$$

则标量链式法则为

$$
\frac{dy}{dx}
=
\frac{dy}{du}\frac{du}{dx}.
$$

因子 $du/dx$ 是操作 $g$ 已知的**局部导数（local derivative）**；$dy/du$ 是从计算图后续部分传来的**上游梯度（upstream gradient）**。二者的乘积就是继续传给 $x$ 的梯度。每个操作都不需要理解整个网络，只需要知道自身的局部导数规则和收到的上游敏感度。

计算图通常是有向无环图，而不是一条简单链。考虑

$$
a=xy, \qquad b=x+y, \qquad f=ab.
$$

变量 $x$ 同时通过 $a$ 和 $b$ 两条路径影响 $f$。多元链式法则会把不同路径的贡献相加：

$$
\frac{\partial f}{\partial x}
=
\frac{\partial f}{\partial a}\frac{\partial a}{\partial x}
+
\frac{\partial f}{\partial b}\frac{\partial b}{\partial x}
=by+a.
$$

同理，$\partial f/\partial y=bx+a$。这正是梯度实现必须在分支处使用 `+=` 的原因。若直接覆盖梯度，就只保留其中一条路径，并且会在没有明显报错的情况下得到错误结果。

计算图结构还解释了三种反复出现的求导模式：

- 加法节点把上游梯度原样传给两个输入；
- 乘法节点用前向传播中另一个输入的值缩放各自的输入梯度；
- 广播操作需要沿前向传播中被虚拟扩展的所有轴对梯度求和。

<details>
<summary><strong>PyTorch：将逐路径链式法则计算与 autograd 对照</strong></summary>

~~~python
import torch

x_value, y_value = 2.0, -3.0

# Forward pass through a graph with two paths from each input to f.
a = x_value * y_value
b = x_value + y_value
f = a * b

# Reverse pass: f = a * b.
df_da = b
df_db = a

# Accumulate contributions through a = x*y and b = x+y.
manual_df_dx = df_da * y_value + df_db * 1.0
manual_df_dy = df_da * x_value + df_db * 1.0

x = torch.tensor(x_value, requires_grad=True)
y = torch.tensor(y_value, requires_grad=True)
output = (x * y) * (x + y)
output.backward()

assert output.item() == f
assert x.grad.item() == manual_df_dx
assert y.grad.item() == manual_df_dy
print("df/dx:", x.grad.item(), "df/dy:", y.grad.item())
~~~

</details>

计算图必须按与依赖关系一致的顺序遍历。前向计算采用从输入到输出的拓扑顺序；反向传播则使用相反顺序，从而保证每个节点在继续向前驱传播之前，已经收到所有下游路径的贡献。

**应用。** 同一规则可处理残差分支、共享参数、绑定嵌入、循环展开和多任务输出头。只要一个张量通过多条路径影响损失，它的最终梯度就是所有路径贡献之和。

**对比总结。** 链式结构会连乘局部导数；分叉会产生多条路径，而路径贡献需要相加；计算图记录了应采用哪种组合规则。反向传播是在这张图上执行动态规划，它复用中间敏感度，而不是分别枚举每一条路径。

### **前向传播与局部导数** {#forward-pass-local-derivatives}

**前向传播（forward pass）**按照依赖顺序计算模型。在启用梯度记录时，它还会保存足够的信息，以便随后执行各个操作的局部反向规则。对于一个两层 MLP，

$$
Z_1=XW_1^{\top}+b_1,
\qquad
H=\operatorname{ReLU}(Z_1),
\qquad
\widehat{Y}=HW_2^{\top}+b_2,
\qquad
\mathcal{L}=\ell(\widehat{Y},Y).
$$

![前向计算图按照依赖顺序，把输入与参数经过一系列操作连接到最终目标。](assets/dl04-forward-computation-graph.svg){fig-align="center" width="78%" fig-alt="一张前向计算图，展示输入与模型参数如何依次经过矩阵乘法、激活、正则化并到达最终目标。"}

*图片来源：[Dive into Deep Learning, Forward Propagation, Backward Propagation, and Computational Graphs](https://d2l.ai/chapter_multilayer-perceptrons/backprop.html)，采用 [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/) 许可。*

不同操作在反向传播中需要不同的已保存信息：

| 前向操作 | 输出 | 反向传播通常需要的信息 |
|---|---|---|
| $y=x_1+x_2$ | 和 | 用于还原广播的输入形状 |
| $y=x_1x_2$ | 积 | 两个输入值 |
| $Y=XW^{\top}+b$ | 仿射输出 | $X$、$W$ 以及 $b$ 的广播形状 |
| $y=\operatorname{ReLU}(x)$ | 阈值化后的值 | 例如 $x>0$ 的掩码 |
| $y=\tanh(x)$ | 有界激活值 | $x$，或已经算出的输出 $y$ |
| 归一化 | 归一化张量 | 统计量、尺度倒数，有时还包括归一化后的激活值 |

保存下来的对象通常称为**缓存（cache）**或**上下文（context）**。保存所有内容可以减少重算，却会增加激活内存；减少保存内容，则可能要求在反向传播时重新执行一部分前向计算。激活检查点（activation checkpointing）正是通过只保留部分边界，并在之后重放内部操作，用额外计算换取更低内存占用。

前向值还决定了当前应启用哪条局部规则。ReLU 的导数取决于输入是否为正；最大池化必须记住获胜索引；对于数据依赖分支，eager 动态图只会记录实际执行过的操作。

<details>
<summary><strong>PyTorch：显式构建前向缓存并统计其中张量</strong></summary>

~~~python
import torch
from torch.nn import functional as F


def mlp_forward_with_cache(x, weight_1, bias_1, weight_2, bias_2):
    """Return predictions plus exactly the values used by a manual backward pass."""
    pre_activation = x @ weight_1.T + bias_1
    hidden = F.relu(pre_activation)
    prediction = hidden @ weight_2.T + bias_2
    cache = {
        "x": x,
        "weight_1": weight_1,
        "pre_activation": pre_activation,
        "hidden": hidden,
        "weight_2": weight_2,
    }
    return prediction, cache


torch.manual_seed(37)
x = torch.randn(16, 8)
weight_1 = torch.randn(12, 8)
bias_1 = torch.randn(12)
weight_2 = torch.randn(4, 12)
bias_2 = torch.randn(4)

prediction, cache = mlp_forward_with_cache(x, weight_1, bias_1, weight_2, bias_2)
cache_bytes = sum(tensor.numel() * tensor.element_size() for tensor in cache.values())

assert prediction.shape == (16, 4)
assert cache["pre_activation"].shape == (16, 12)
print("cached tensors:", {name: tuple(value.shape) for name, value in cache.items()})
print("cache payload bytes:", cache_bytes)
~~~

</details>

这个字节数包含了已经由其他对象持有的张量，因此不能把它当作峰值内存分析器。不过，它清楚揭示了反向公式依赖哪些信息。框架可能保存视图、压缩掩码、融合后的中间量或可重算元数据，而不一定采用这里展示的 Python 字典形式。

**应用。** 理解保存张量可以解释：为什么训练比推理更占内存，为什么原地修改可能破坏反向传播，以及检查点技术或融合自定义 kernel 如何改变内存与计算之间的权衡。

**对比总结。** 普通前向计算负责得到数值；启用梯度的前向计算还会构建导数历史并保留上下文。缓存减少反向计算量但消耗内存；重算降低内存，却需要再执行一次部分前向传播。

### **反向模式的反向传播** {#reverse-mode-backpropagation}

对于每个中间变量 $v$，定义它的**伴随量（adjoint）**或反向敏感度：

$$
\bar{v}=\frac{\partial \mathcal{L}}{\partial v}.
$$

反向模式从标量损失开始，以 $\bar{\mathcal{L}}=1$ 作为种子，并按逆拓扑顺序访问操作。若一个节点计算 $v=f(u_1,\ldots,u_k)$，它的 pullback 执行

$$
\bar{u}_i
\mathrel{+}=
\left(\frac{\partial v}{\partial u_i}\right)^{\top}\bar{v}.
$$

对于向量中间量，转置符号非常重要。从概念上看，节点接收一个位于输出空间中的敏感度，并把它拉回到每个输入空间。算法可概括为：

```text
1. Execute the forward graph and retain required contexts.
2. Topologically order nodes from leaves to the scalar loss.
3. Initialize every adjoint to zero and set loss.adjoint = 1.
4. Visit nodes in reverse topological order.
5. Apply each node's local pullback and accumulate into parent adjoints.
6. Read the adjoints of parameters and requested inputs.
```

对上一节的 MLP 和均方误差，令 $G_{\widehat{Y}}=\partial\mathcal{L}/\partial\widehat{Y}$，反向方程为

$$
G_{W_2}=G_{\widehat{Y}}^{\top}H,
\quad
G_{b_2}=\sum_{b=1}^{B}G_{\widehat{Y},b},
\quad
G_H=G_{\widehat{Y}}W_2,
$$

$$
G_{Z_1}=G_H\odot\mathbf{1}[Z_1>0],
\quad
G_{W_1}=G_{Z_1}^{\top}X,
\quad
G_{b_1}=\sum_{b=1}^{B}G_{Z_1,b}.
$$

每个梯度的形状都与被求导对象一致：$G_{W_1}$ 的形状是 `[H, D_in]`，$G_{b_1}$ 的形状是 `[H]`，$G_X$ 的形状是 `[B, D_in]`。检查形状是排查手动反向推导错误的最快方法之一。

<details>
<summary><strong>PyTorch：手动推导完整的两层 MLP 反向传播</strong></summary>

~~~python
import torch
from torch.nn import functional as F

torch.manual_seed(41)
B, D_IN, HIDDEN, D_OUT = 5, 3, 7, 2

x = torch.randn(B, D_IN)
target = torch.randn(B, D_OUT)
weight_1 = torch.randn(HIDDEN, D_IN)
bias_1 = torch.randn(HIDDEN)
weight_2 = torch.randn(D_OUT, HIDDEN)
bias_2 = torch.randn(D_OUT)

# Forward pass.
z_1 = x @ weight_1.T + bias_1
hidden = F.relu(z_1)
prediction = hidden @ weight_2.T + bias_2
loss = F.mse_loss(prediction, target, reduction="mean")

# Reverse pass. MSE averages over every prediction element.
grad_prediction = 2.0 * (prediction - target) / prediction.numel()
grad_weight_2 = grad_prediction.T @ hidden
grad_bias_2 = grad_prediction.sum(dim=0)
grad_hidden = grad_prediction @ weight_2
grad_z_1 = grad_hidden * (z_1 > 0)
grad_weight_1 = grad_z_1.T @ x
grad_bias_1 = grad_z_1.sum(dim=0)
grad_x = grad_z_1 @ weight_1

# Rebuild the same graph with tracked leaves and compare against autograd.
x_ref = x.clone().requires_grad_(True)
w1_ref = weight_1.clone().requires_grad_(True)
b1_ref = bias_1.clone().requires_grad_(True)
w2_ref = weight_2.clone().requires_grad_(True)
b2_ref = bias_2.clone().requires_grad_(True)
prediction_ref = F.relu(x_ref @ w1_ref.T + b1_ref) @ w2_ref.T + b2_ref
loss_ref = F.mse_loss(prediction_ref, target)
loss_ref.backward()

comparisons = [
    (grad_x, x_ref.grad),
    (grad_weight_1, w1_ref.grad),
    (grad_bias_1, b1_ref.grad),
    (grad_weight_2, w2_ref.grad),
    (grad_bias_2, b2_ref.grad),
]
assert all(torch.allclose(manual, automatic, atol=1e-6) for manual, automatic in comparisons)
print("manual backward matches autograd:", True)
~~~

</details>

反向传播之后，许多 eager 框架会释放已保存的计算图状态，因为这些状态可能非常大。`retain_graph=True` 会保留计算图以便再次反向遍历；`create_graph=True` 则记录导数计算本身，使其还能继续求高阶导数。两者解决的问题不同，也都会增加内存占用。

**应用。** 反向模式支撑常规神经网络训练、显著性梯度、基于梯度的输入优化，以及任何“标量目标对大量参数求导”的任务。

**对比总结。** 前向计算把数值从输入推向损失；反向模式把一个损失敏感度拉回到所有祖先节点。梯度累加处理共享路径，缓存的局部上下文则避免重复进行符号展开。

### **Jacobian、向量-Jacobian 积与 Jacobian-向量积** {#jacobians-vjp-jvp}

对于向量函数 $f:\mathbb{R}^{n}\rightarrow\mathbb{R}^{m}$，Jacobian 定义为

$$
J_f(x)
=
\frac{\partial f}{\partial x}
\in\mathbb{R}^{m\times n},
\qquad
[J_f]_{ij}=\frac{\partial f_i}{\partial x_j}.
$$

显式构造 $J_f$ 往往非常浪费。若某层把一百万维激活映射到另一个一百万维激活，其形式上的 Jacobian 将包含 $10^{12}$ 个元素，即使矩阵本身具有稀疏结构，或其作用可以被廉价计算。自动微分系统因此通常暴露 Jacobian 的**乘积**，而不是完整矩阵。

**Jacobian-向量积（Jacobian-vector product, JVP）**把输入切向量 $u\in\mathbb{R}^{n}$ 向前推送：

$$
J_f(x)u\in\mathbb{R}^{m}.
$$

它就是 $f$ 沿 $u$ 方向的方向导数：

$$
J_f(x)u
=
\left.\frac{d}{d\epsilon}f(x+\epsilon u)\right|_{\epsilon=0}.
$$

**向量-Jacobian 积（vector-Jacobian product, VJP）**把输出余切向量 $v\in\mathbb{R}^{m}$ 向后拉回：

$$
v^{\top}J_f(x)\in\mathbb{R}^{n}.
$$

当 $m=1$ 且 $v=1$ 时，VJP 就是熟悉的标量输出梯度。对非标量输出调用 PyTorch 的 `.backward(gradient=...)` 同样是在计算 VJP：传入的张量就是输出端的余切种子。

<details>
<summary><strong>PyTorch：在不显式构造 Jacobian 的情况下验证 JVP 与 VJP</strong></summary>

~~~python
import torch
from torch.func import jacrev, jvp, vjp


def vector_function(x: torch.Tensor) -> torch.Tensor:
    return torch.stack(
        (
            x[0] * x[1] + torch.sin(x[2]),
            x[0].square() + torch.exp(x[1]),
        )
    )


x = torch.tensor([0.7, -0.4, 1.2], dtype=torch.float64)
input_tangent = torch.tensor([1.0, 2.0, -1.0], dtype=torch.float64)
output_cotangent = torch.tensor([0.5, -1.5], dtype=torch.float64)

output, jvp_result = jvp(vector_function, (x,), (input_tangent,))
output_again, pullback = vjp(vector_function, x)
vjp_result = pullback(output_cotangent)[0]

# The full Jacobian is formed only as a small reference check.
jacobian = jacrev(vector_function)(x)
assert jacobian.shape == (2, 3)
assert torch.allclose(jvp_result, jacobian @ input_tangent)
assert torch.allclose(vjp_result, output_cotangent @ jacobian)
assert torch.allclose(output, output_again)

print("JVP shape:", tuple(jvp_result.shape))
print("VJP shape:", tuple(vjp_result.shape))
~~~

</details>

术语 *tangent* 与 *cotangent* 描述这些对象在数学上如何变换。在代码中，两者都表现为与相应 primal 变量形状一致的张量；但在推导转置、批处理导数变换或组合自定义规则时，这一区分非常重要。

**应用。** JVP 可用于方向敏感度、隐式模型、微分方程，以及 forward-over-reverse 的 Hessian 乘积；VJP 可用于标量损失训练、伴随方法，以及对选定输出加权组合求梯度。

**对比总结。** 完整 Jacobian 以高存储成本回答所有输入-输出偏导；JVP 询问输出沿一个输入方向如何变化；VJP 询问一个加权输出敏感度如何映射回输入。现代自动微分通常只计算任务真正需要的乘积。

### **前向模式与反向模式微分** {#forward-mode-vs-reverse-mode}

前向模式与反向模式沿相反的计算方向应用同一链式法则。假设计算 $f:\mathbb{R}^{n}\to\mathbb{R}^{m}$ 的代价为 $C$。

| 模式 | 传播的量 | 构造完整 Jacobian 所需的近似遍历次数 | 最适合的问题形状 |
|---|---|---:|---|
| 前向模式 | 通过 JVP 传播一个输入切向量 | $n$ | 输入少、输出多 |
| 反向模式 | 通过 VJP 传播一个输出余切向量 | $m$ | 输入多、输出少 |

一次前向模式遍历能计算一个输入方向对全部输出的影响；一次反向模式遍历则能计算所有输入对一个加权输出方向的影响。神经网络训练通常拥有数百万个参数输入，却只有一个标量损失，因此反向模式大约只需一次前向和一次同量级反向计算，就能获得全部参数梯度，而不必为每个参数分别遍历。

相应代价是内存。前向模式可让切向量与 primal 值一起传播，而无需保存完整的反向 tape；反向模式通常必须保留或重算中间上下文，直到反向传播使用它们。实际成本取决于基本操作结构、批处理、编译器融合和检查点策略，但输入与输出维度仍是选择模式时最重要的第一判断。

不同微分模式还可以组合。对于标量函数 $g:\mathbb{R}^{n}\to\mathbb{R}$，其 Hessian $H_g\in\mathbb{R}^{n\times n}$ 往往大到无法构造。Hessian-向量积

$$
H_g(x)u

$$

可以通过对反向模式产生的梯度函数执行一次 JVP 来获得。这种 **forward-over-reverse** 组合可用于二阶优化、曲率诊断、影响方法和隐式微分。

<details>
<summary><strong>PyTorch：通过组合变换计算 Hessian-向量积</strong></summary>

~~~python
import torch
from torch.func import grad, hessian, jvp


def scalar_objective(x: torch.Tensor) -> torch.Tensor:
    interaction = 0.1 * x.sum().square()
    return (torch.sin(x) * x.square()).sum() + interaction


x = torch.tensor([0.2, -0.7, 1.1], dtype=torch.float64)
direction = torch.tensor([1.0, 0.5, -2.0], dtype=torch.float64)

gradient_function = grad(scalar_objective)  # reverse mode: R^n -> R^n
gradient_value, hessian_vector = jvp(gradient_function, (x,), (direction,))

# Construct the tiny full Hessian only to verify the product.
full_hessian = hessian(scalar_objective)(x)
assert gradient_value.shape == x.shape
assert torch.allclose(hessian_vector, full_hessian @ direction, atol=1e-10)
print("Hessian-vector product:", hessian_vector)
~~~

</details>

在常规 autograd 代码中，若返回的梯度本身还需要继续可微，就必须设置 `create_graph=True`。`retain_graph=True` 仅仅保留现有 tape 以便再次遍历，并不会自动让导数操作变得可微。混淆这两个标志既会导致正确性问题，也会造成不必要的内存增长。

**应用。** 反向模式是标量损失模型训练的默认方案；前向模式适合低维参数敏感度或宽输出模拟器；混合模式则让高级优化和科学机器学习无需显式构造完整 Hessian 或 Jacobian。

**对比总结。** 模式选择主要由输入与输出维度决定。反向模式并非始终更快；高阶导数最好表示为 JVP 与 VJP 变换的组合，而不是显式导数矩阵。

### **常见神经网络层的梯度** {#gradients-common-neural-network-layers}

反向传播之所以实用，是因为框架为可复用的基本操作实现了局部 pullback。不同架构中会反复出现以下模式。

对于仿射层

$$
Y=XW^{\top}+b,
\quad
X\in\mathbb{R}^{B\times D_{in}},
\quad
W\in\mathbb{R}^{D_{out}\times D_{in}},
$$

以及上游梯度 $G=\partial\mathcal{L}/\partial Y$，

$$
\frac{\partial\mathcal{L}}{\partial X}=GW,
\qquad
\frac{\partial\mathcal{L}}{\partial W}=G^{\top}X,
\qquad
\frac{\partial\mathcal{L}}{\partial b}=\sum_{i=1}^{B}G_i.
$$

偏置梯度需要归约，是因为前向传播把一个偏置向量广播到了整个 batch。

逐元素激活函数会把上游梯度与局部导数逐元素相乘：

$$
G_x=G_y\odot\phi'(x).
$$

对于 ReLU，在常用的零点约定下，$\phi'(x)=\mathbf{1}[x>0]$。对于 sigmoid 输出 $s=\sigma(x)$，$\phi'(x)=s(1-s)$；对于 $t=\tanh(x)$，$\phi'(x)=1-t^2$。复用前向输出可以避免再次计算非线性函数。

softmax 与 cross-entropy 最适合视为一个融合表达式来求导。对 logits $z$、one-hot 目标 $q$ 以及概率 $p=\operatorname{softmax}(z)$，

$$
\ell=-\sum_{k=1}^{K}q_k\log p_k,
\qquad
\frac{\partial\ell}{\partial z}=p-q.
$$

这个紧凑结果避免了显式保存稠密 softmax Jacobian。若对 $B$ 个样本取平均，还需除以 $B$。

其他层遵循结构化的累加规则。embedding 的反向传播把梯度 scatter-add 到每个被引用的行，因此重复 ID 会发生累加；卷积的反向传播是另一种类似卷积或相关的操作，并把不同位置对共享权重的贡献累加；归一化的反向传播会耦合共享同一均值和方差的数值，因此不能把它视为相互独立的逐元素缩放。

<details>
<summary><strong>PyTorch：推导 softmax-cross-entropy 与仿射层梯度</strong></summary>

~~~python
import torch
from torch.nn import functional as F

torch.manual_seed(43)
B, D_IN, K = 6, 4, 3
x = torch.randn(B, D_IN, dtype=torch.float64)
weight = torch.randn(K, D_IN, dtype=torch.float64)
bias = torch.randn(K, dtype=torch.float64)
targets = torch.tensor([0, 2, 1, 1, 0, 2])

logits = x @ weight.T + bias
probabilities = logits.softmax(dim=1)

# Fused cross-entropy derivative: (probability - one_hot_target) / B.
grad_logits = probabilities.clone()
grad_logits[torch.arange(B), targets] -= 1.0
grad_logits /= B

manual_grad_weight = grad_logits.T @ x
manual_grad_bias = grad_logits.sum(dim=0)
manual_grad_x = grad_logits @ weight

x_ref = x.clone().requires_grad_(True)
weight_ref = weight.clone().requires_grad_(True)
bias_ref = bias.clone().requires_grad_(True)
loss = F.cross_entropy(x_ref @ weight_ref.T + bias_ref, targets)
loss.backward()

assert torch.allclose(manual_grad_weight, weight_ref.grad, atol=1e-10)
assert torch.allclose(manual_grad_bias, bias_ref.grad, atol=1e-10)
assert torch.allclose(manual_grad_x, x_ref.grad, atol=1e-10)
print("fused derivative matches autograd:", True)
~~~

</details>

自定义 kernel 必须为每个可微输入返回一个梯度，正确归约广播维度，并维持 dtype 与 device 约定。即使局部导数看似合理，只要形状或累加语义错误，整体梯度仍然会出错。

**应用。** 熟悉这些规则有助于推导自定义层、审计融合 kernel、估算反向计算成本，并判断为什么某些张量必须保存；在调试时，autograd 的输出也会因此不再像黑箱。

**对比总结。** 逐元素层会掩蔽或缩放上游梯度；仿射层使用矩阵收缩；广播参数需要归约扩展轴；共享参数会累加每次使用的贡献；融合损失利用代数化简避免完整 Jacobian。

### **从零实现微型自动微分引擎** {#micro-autograd-engine}

一个最小反向模式引擎只需要四个核心概念：

1. 一个 value 保存前向数值和累积梯度；
2. 每个操作创建一个与其父节点相连的输出节点；
3. 输出节点拥有局部 `_backward` 函数，用于更新父节点梯度；
4. `.backward()` 构建拓扑顺序，把最终梯度设为 1，再按相反顺序执行局部规则。

![Micrograd 把一个标量神经元可视化为动态图，其中节点同时保存前向数值与反向梯度。](assets/micrograd-neuron-graph.svg){fig-align="center" width="100%" fig-alt="一个较宽的 Micrograd 双输入神经元计算图，展示标量操作、前向数值和梯度。"}

*图片来源：Andrej Karpathy，[Micrograd](https://github.com/karpathy/micrograd) 的 `gout.svg`，MIT License。*

下面的引擎只处理标量，因此每个局部导数都清晰可见。张量框架采用相同架构，只是它们的向量化 kernel 会在 pullback 中执行矩阵、归约、scatter 或卷积操作。

<details>
<summary><strong>Python：实现标量反向模式自动微分</strong></summary>

~~~python
import math
import torch


class Value:
    """A scalar value with a dynamically constructed reverse-mode graph."""

    def __init__(self, data, children=(), operation="", label=""):
        self.data = float(data)
        self.grad = 0.0
        self.parents = tuple(children)
        self.operation = operation
        self.label = label
        self._backward = lambda: None

    @staticmethod
    def _coerce(other):
        return other if isinstance(other, Value) else Value(other)

    def __add__(self, other):
        other = self._coerce(other)
        output = Value(self.data + other.data, (self, other), "+")

        def backward():
            self.grad += output.grad
            other.grad += output.grad

        output._backward = backward
        return output

    def __mul__(self, other):
        other = self._coerce(other)
        output = Value(self.data * other.data, (self, other), "*")

        def backward():
            self.grad += other.data * output.grad
            other.grad += self.data * output.grad

        output._backward = backward
        return output

    def __pow__(self, exponent):
        output = Value(self.data**exponent, (self,), f"**{exponent}")

        def backward():
            self.grad += exponent * self.data ** (exponent - 1) * output.grad

        output._backward = backward
        return output

    def tanh(self):
        value = math.tanh(self.data)
        output = Value(value, (self,), "tanh")

        def backward():
            self.grad += (1.0 - value**2) * output.grad

        output._backward = backward
        return output

    def __neg__(self):
        return self * -1.0

    def __sub__(self, other):
        return self + (-self._coerce(other))

    def __radd__(self, other):
        return self + other

    def __rmul__(self, other):
        return self * other

    def backward(self):
        topological_order = []
        visited = set()

        def build(node):
            if node not in visited:
                visited.add(node)
                for parent in node.parents:
                    build(parent)
                topological_order.append(node)

        build(self)
        self.grad = 1.0
        for node in reversed(topological_order):
            node._backward()


# A two-input tanh neuron: y = tanh(x1*w1 + x2*w2 + b).
x_1 = Value(1.5, label="x1")
x_2 = Value(-2.0, label="x2")
w_1 = Value(0.7, label="w1")
w_2 = Value(-0.3, label="w2")
bias = Value(0.2, label="b")
output = (x_1 * w_1 + x_2 * w_2 + bias).tanh()
output.backward()

# Compare all leaf gradients against the equivalent PyTorch graph.
torch_leaves = [torch.tensor(v.data, requires_grad=True) for v in (x_1, x_2, w_1, w_2, bias)]
tx1, tx2, tw1, tw2, tbias = torch_leaves
torch_output = torch.tanh(tx1 * tw1 + tx2 * tw2 + tbias)
torch_output.backward()

micrograd_leaves = (x_1, x_2, w_1, w_2, bias)
assert all(abs(value.grad - tensor.grad.item()) < 1e-6 for value, tensor in zip(micrograd_leaves, torch_leaves))
print("output:", round(output.data, 6))
print("leaf gradients:", [round(value.grad, 6) for value in micrograd_leaves])
~~~

</details>

这里有两个容易遗漏的实现细节。第一，梯度必须使用 `+=`，因为同一节点可能被重复使用。第二，拓扑排序保证节点的 pullback 执行前，它的输出梯度已经收齐所有贡献。对于包含共享子表达式的图，若不使用记忆机制而递归展开表达式，计算量可能呈指数增长。

生产级引擎还需要处理张量形状、device、dtype 提升、view、修改与版本检查、并行调度、自定义 kernel、图剪枝、saved-tensor hook、编译执行和高阶微分。这个小型引擎并不是替代品；它只是把这些系统背后的不变量单独呈现出来。

**应用。** 从零构建标量引擎，可以把 autograd 从一个不透明服务转化为具体的图算法。实现自定义 `autograd.Function` 或排查梯度漏累加时，它也可作为简洁的参照实现。

**对比总结。** 手动微分要为一个具体模型编写一套反向程序；微型引擎把可复用的 pullback 绑定到基本操作；生产级张量引擎则用向量化 kernel 和系统级状态管理实现同一套图逻辑。

### **梯度检查** {#gradient-checking}

梯度检查把解析梯度或自动微分梯度与数值近似进行比较。对于标量函数 $f$ 和坐标 $i$，中心差分为

$$
\frac{\partial f}{\partial x_i}
\approx
\frac{f(x+\epsilon e_i)-f(x-\epsilon e_i)}{2\epsilon}.
$$

中心差分的截断误差阶为 $O(\epsilon^2)$，但把 $\epsilon$ 设得任意小会加剧浮点相消。可靠的检查通常使用 `float64`、如 $10^{-6}$ 的适中 epsilon、确定性计算，同时检查相对误差和绝对误差：

$$
\operatorname{relative\ error}
=
\frac{|g_{analytic}-g_{numeric}|}
{\max(1,|g_{analytic}|,|g_{numeric}|)}.
$$

对于高维输入，逐坐标检查的成本很高。方向检查比较

$$
\nabla f(x)^{\top}u
\quad\text{与}\quad
\frac{f(x+\epsilon u)-f(x-\epsilon u)}{2\epsilon},
$$

它只用两次函数求值，就能沿一个随机方向同时检验全部坐标。不过，某个与该方向特殊对齐的错误可能被漏掉，因此使用多个方向或抽样多个坐标会更可靠。

检查时应避开不可微边界。ReLU 在零点、max 的并列值、离散索引决策、裁剪阈值和随机采样，都可能产生合理但依赖约定的差异。dropout 与随机增强则应关闭或固定随机状态。

<details>
<summary><strong>PyTorch：用 gradcheck 和方向导数验证自定义反向规则</strong></summary>

~~~python
import torch
from torch.autograd import Function, gradcheck


class CubicWithBias(Function):
    @staticmethod
    def forward(ctx, x, bias):
        ctx.save_for_backward(x)
        return x**3 + bias

    @staticmethod
    def backward(ctx, grad_output):
        (x,) = ctx.saved_tensors
        grad_x = grad_output * 3.0 * x.square()
        grad_bias = grad_output.sum()  # bias was broadcast over x
        return grad_x, grad_bias


def objective(x, bias):
    return torch.sin(CubicWithBias.apply(x, bias)).sum()


x = torch.tensor([0.2, -0.8, 1.3], dtype=torch.float64, requires_grad=True)
bias = torch.tensor(0.15, dtype=torch.float64, requires_grad=True)

# gradcheck perturbs each double-precision input and compares Jacobian entries.
assert gradcheck(CubicWithBias.apply, (x, bias), eps=1e-6, atol=1e-5, rtol=1e-4)

# A separate directional check validates the complete scalar objective.
direction = torch.tensor([1.0, -0.5, 2.0], dtype=torch.float64)
analytical_gradient = torch.autograd.grad(objective(x, bias), x)[0]
analytical_directional = analytical_gradient @ direction

epsilon = 1e-6
with torch.no_grad():
    numerical_directional = (
        objective(x + epsilon * direction, bias)
        - objective(x - epsilon * direction, bias)
    ) / (2.0 * epsilon)

assert torch.allclose(analytical_directional, numerical_directional, atol=1e-7, rtol=1e-6)
print("directional derivative error:", float((analytical_directional - numerical_directional).abs()))
~~~

</details>

数值检查通过会提高我们对局部导数规则的信心，但不能证明模型在语义上正确。损失函数可能用了错误标签，张量可能沿错误轴对齐，甚至解析与数值代码可能共同继承了同一个上游错误。

**应用。** 梯度检查最适合新的自定义操作、非常规广播、隐式求解器、可微渲染和手动推导损失。它应在测试中使用极小且确定性的输入运行，而不应放入常规训练循环。

**对比总结。** autograd 根据局部规则高效计算导数；有限差分通过重复求值近似导数；梯度检查比较二者。数值一致性验证局部微积分，形状与模型语义则必须由其他测试验证。

### **梯度消失与梯度爆炸** {#vanishing-exploding-gradients}

在深层复合函数

$$
h^{(l)}=f_l(h^{(l-1)}),
$$

中，从第 $L$ 层到较早第 $l$ 层的敏感度包含一串 Jacobian 乘积：

$$
\frac{\partial h^{(L)}}{\partial h^{(l)}}
=
J_LJ_{L-1}\cdots J_{l+1}.
$$

若相关奇异值反复小于 1，梯度范数往往指数衰减；若反复大于 1，梯度则可能指数增长。方向与大小同样重要，所以只用特征值或标量来理解只是近似，但反复相乘的机制是根本原因。

![sigmoid 只有在零附近具有明显导数，在两侧饱和区域中的导数都接近零。](assets/dl04-sigmoid-gradient.svg){fig-align="center" width="68%" fig-alt="对比 sigmoid 函数值与其导数的曲线，展示输入为较大正值或负值时导数趋近于零。"}

*图片来源：[Dive into Deep Learning, Numerical Stability and Initialization](https://d2l.ai/chapter_multilayer-perceptrons/numerical-stability-and-init.html)，采用 [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/) 许可。*

sigmoid 清楚展示了激活函数带来的影响：$\sigma'(x)=\sigma(x)(1-\sigma(x))\leq 1/4$，并且导数在两侧饱和区域都趋近于零。反复乘上这类因子，会阻止信用传到较早层。权重矩阵还可能进一步放大或收缩信号。循环网络在许多时间步重复使用同一个状态转移，因此同一种不稳定性可能连续重复数百次。

不同问题具有不同症状：

| 问题 | 可观察症状 | 后果 |
|---|---|---|
| 梯度消失 | 早期层范数接近零；特征几乎不变 | 长程依赖学习缓慢或完全失败 |
| 梯度爆炸 | 范数突然增大、损失尖峰、出现 Inf/NaN | 更新破坏模型并导致数值溢出 |
| 死亡或断开路径 | 梯度严格为零或 `None` | 参数无法从目标获得信用 |
| 条件不良 | 不同方向或层的范数差异很大 | 训练不稳定且对步长敏感 |

不同缓解方法针对不同原因。Xavier 或 He 初始化控制初始方差；非饱和激活保留有效局部斜率；残差连接增加恒等路径；归一化调节激活尺度；LSTM/GRU 门控建立可控记忆路径；梯度裁剪限制爆炸更新；更短的有效路径则减少连续乘积次数。裁剪只处理幅值症状，无法修复根本性的梯度消失或图路径断开。

<details>
<summary><strong>PyTorch：观察普通标量链和残差标量链中的梯度乘积</strong></summary>

~~~python
import torch


def plain_chain_gradient(scale: float, depth: int) -> float:
    x = torch.tensor(1.0, dtype=torch.float64, requires_grad=True)
    hidden = x
    for _ in range(depth):
        hidden = scale * hidden
    return torch.autograd.grad(hidden, x)[0].item()


def residual_chain_gradient(scale: float, depth: int) -> float:
    x = torch.tensor(0.5, dtype=torch.float64, requires_grad=True)
    hidden = x
    for _ in range(depth):
        # Each local derivative contains an identity contribution of 1.
        hidden = hidden + scale * torch.tanh(hidden)
    return torch.autograd.grad(hidden, x)[0].item()


depth = 40
vanishing = plain_chain_gradient(scale=0.5, depth=depth)
exploding = plain_chain_gradient(scale=1.2, depth=depth)
residual = residual_chain_gradient(scale=0.01, depth=depth)

assert abs(vanishing - 0.5**depth) < 1e-15
assert abs(exploding - 1.2**depth) < 1e-8
assert 1.0 < residual < 2.0

print("plain scale 0.5:", f"{vanishing:.3e}")
print("plain scale 1.2:", f"{exploding:.3e}")
print("small residual updates:", f"{residual:.3f}")
~~~

</details>

真实网络需要测量，而不能只根据损失曲线猜测。应监控逐层梯度范数、参数与更新的比例、激活分布和非有限值，并比较多个训练步骤与随机种子。ReLU 或 masking 后偶尔出现一个零值可能完全正常；若整层长期接近零，才更可能表示结构性问题。

**应用。** 对非常深的网络、循环与状态空间模型、长上下文系统、混合精度训练，以及缺乏成熟初始化方案的自定义架构而言，梯度流诊断都是必需环节。

**对比总结。** 梯度消失与爆炸源于反复的 Jacobian 乘积；断开梯度来自计算图结构；非有限梯度还可能来自不稳定数值运算。初始化、架构、归一化、裁剪和数值精度分别解决问题的不同部分。

### **本章对比与总结** {#chapter-comparison-summary}

理解反向传播的最佳方式，是把它看作运输局部敏感度的图算法，而不是一个庞大的符号导数。前向执行产生数值和导数上下文；反向执行接收一个输出敏感度，并把贡献累加到每个祖先节点。

| 概念 | 核心对象 | 方向 | 主要资源成本 | 常见错误 |
|---|---|---|---|---|
| 链式法则 | 局部导数与上游梯度 | 沿依赖关系组合 | 每条图边上的算术运算 | 忘记累加分支贡献 |
| 前向传播 | primal 值与已保存上下文 | 输入到输出 | 激活计算与存储 | 丢弃或修改反向所需数值 |
| 反向模式 | 伴随量 / VJP | 输出到输入 | 保存的激活与 pullback | 错误的种子、形状或累加 |
| 前向模式 | 切向量 / JVP | 输入到输出 | 每个方向执行一次切向传播 | 对数百万个独立输入使用该模式 |
| 完整 Jacobian | 全部输出-输入偏导 | 同时暴露两个维度 | 可能需要 $O(mn)$ 存储 | 只需矩阵乘积时仍显式构造矩阵 |
| 微型 autograd | 动态 DAG 与局部闭包 | 逆拓扑 | 每个基本操作对应一个标量节点 | 忘记拓扑顺序或 `+=` |
| 梯度检查 | 有限差分估计 | 重复前向求值 | 昂贵但简单 | 在随机或不可微点上检查 |
| 梯度流诊断 | 范数与有限值检查 | 跨层或跨时间 | 监控开销 | 把裁剪当成万能修复方法 |

本章的主要结论是：

1. 梯度是指定标量目标的局部敏感度，不是因果解释，也不保证一次有限更新必然改善目标。
2. 局部导数沿路径相乘；多个分支共享同一祖先时，路径贡献相加。
3. 反向模式把标量损失的种子设为 1，并按逆拓扑顺序传播 VJP。
4. 前向缓存减少重算，却使训练内存高于推理内存。
5. JVP 与 VJP 无需显式构造完整 Jacobian，就能计算它对向量的作用。
6. 反向模式适合多参数、标量损失训练；前向模式适合低输入维、宽输出敏感度；混合模式可计算高阶乘积。
7. 层的 pullback 必须准确复现前向传播中的广播、参数共享和归约行为。
8. 小型 autograd 引擎需要图构建、局部闭包、拓扑排序、梯度种子和梯度累加。
9. 有限差分是一种诊断参照，其可靠性取决于精度、epsilon、确定性和求值点的平滑性。
10. 梯度消失、爆炸、断开和非有限值具有不同原因，必须采用不同干预方式。

第 05 章将把正确梯度转化为完整学习过程：它会定义损失函数、初始化与更新规则、优化器状态、学习率调度、混合精度，以及决定重复更新能否收敛的训练动态。